# Tinker Email Agent with Separate SFT & RLHF Datasets (340 Total) (aspired by ART OPENPIPE)

This notebook implements an email search agent using:
1. **SFT Warmup**: 150 dedicated scenarios for supervised fine-tuning
2. **RLHF Training**: 150 DIFFERENT scenarios for reinforcement learning
3. **Validation**: 40 held-out scenarios for evaluation

In [1]:
# Tinker ART Email Agent with Separate SFT & RLHF Datasets (340 Total)
"""
This notebook implements an email search agent using:
1. **SFT Warmup**: 150 dedicated scenarios for supervised fine-tuning (3 epochs)
2. **RLHF Training**: 150 DIFFERENT scenarios for reinforcement learning
3. **Validation**: 40 held-out scenarios for evaluation

**Total dataset: 340 scenarios with NO OVERLAP between phases**
- SFT: 150 scenarios (only for supervised learning)
- RLHF: 150 different scenarios (only for reinforcement learning)
- Validation: 40 scenarios (for evaluation)
"""

'\nThis notebook implements an email search agent using:\n1. **SFT Warmup**: 150 dedicated scenarios for supervised fine-tuning (3 epochs)\n2. **RLHF Training**: 150 DIFFERENT scenarios for reinforcement learning\n3. **Validation**: 40 held-out scenarios for evaluation\n\n**Total dataset: 340 scenarios with NO OVERLAP between phases**\n- SFT: 150 scenarios (only for supervised learning)\n- RLHF: 150 different scenarios (only for reinforcement learning)\n- Validation: 40 scenarios (for evaluation)\n'

In [1]:
!pip install -q "tinker" "tinker_cookbook" "datasets" "python-dotenv" "tqdm"

In [2]:
#!pip uninstall -y tinker-cookbook
!git clone https://github.com/thinking-machines-lab/tinker-cookbook.git
!pip install -e tinker-cookbook

fatal: destination path 'tinker-cookbook' already exists and is not an empty directory.
Obtaining file:///content/tinker-cookbook
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tinker_cookbook (pyproject.toml) ... done
  Created wheel for tinker_cookbook: filename=tinker_cookbook-0.1.0-py3-none-any.whl size=7710 sha256=0dfb71fc18d615f64e5a12a1dc371bfb0843ad9edf7b645ae02bba047b51de2d
  Stored in directory: /tmp/pip-ephem-wheel-cache-fgoscne3/wheels/5e/e4/ee/e549b17cc3c04e3d32a1eec323b99dddf4dda2379138fe63d8
Successfully built tinker_cookbook
  Attempting uninstall: tinker_cookbook
    Found existing installation: tinker_cookbook 0.1.0
    Uninstalling tinker_cookbook-0.1.0:
      Successfully uninstalled tinker_cookbook-0.1.0


In [3]:
# re-run the notebook after above execution finished

In [4]:
# ============================================================
# Email Search Agent with Tinker RLHF
# ============================================================

import os
import json
import sqlite3
import random
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Literal

from datasets import Dataset, Features, Sequence, Value, load_dataset
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

In [6]:
# -------------------------
# Environment variables
# -------------------------

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if not os.environ.get("TINKER_API_KEY"):
    os.environ["TINKER_API_KEY"] = ""  # TODO: put your key here or set env in Colab UI

if not os.environ["TINKER_API_KEY"]:
    raise ValueError("TINKER_API_KEY is required to talk to Tinker.")

In [7]:
# -------------------------
# Email / scenario models
# -------------------------

class Email(BaseModel):
    message_id: str
    date: str  # 'YYYY-MM-DD HH:MM:SS'
    subject: Optional[str] = None
    from_address: Optional[str] = None
    to_addresses: List[str] = []
    cc_addresses: List[str] = []
    bcc_addresses: List[str] = []
    body: Optional[str] = None
    file_name: Optional[str] = None

class Scenario(BaseModel):
    id: int
    question: str
    answer: str
    message_ids: List[str]
    how_realistic: float
    inbox_address: str
    query_date: str
    split: Literal["train", "test"]

@dataclass
class SearchResult:
    message_id: str
    snippet: str

class FinalAnswer(BaseModel):
    answer: str
    source_ids: list[str]

DB_PATH = "./enron_emails.db"
EMAIL_DATASET_REPO_ID = "corbt/enron-emails"
SCENARIO_DATASET_REPO_ID = "corbt/enron_emails_sample_questions"

db_conn = None  # global

def create_email_database():
    print("Creating email DB from Hugging Face dataset...")

    SQL_CREATE_TABLES = """
    DROP TABLE IF EXISTS recipients;
    DROP TABLE IF EXISTS emails_fts;
    DROP TABLE IF EXISTS emails;

    CREATE TABLE emails (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        message_id TEXT UNIQUE,
        subject TEXT,
        from_address TEXT,
        date TEXT,
        body TEXT,
        file_name TEXT
    );

    CREATE TABLE recipients (
        email_id TEXT,
        recipient_address TEXT,
        recipient_type TEXT
    );
    """

    SQL_CREATE_INDEXES_TRIGGERS = """
    CREATE INDEX idx_emails_from ON emails(from_address);
    CREATE INDEX idx_emails_date ON emails(date);
    CREATE INDEX idx_emails_message_id ON emails(message_id);
    CREATE INDEX idx_recipients_address ON recipients(recipient_address);
    CREATE INDEX idx_recipients_type ON recipients(recipient_type);
    CREATE INDEX idx_recipients_email_id ON recipients(email_id);
    CREATE INDEX idx_recipients_address_email ON recipients(recipient_address, email_id);

    CREATE VIRTUAL TABLE emails_fts USING fts5(
        subject,
        body,
        content='emails',
        content_rowid='id'
    );

    CREATE TRIGGER emails_ai AFTER INSERT ON emails BEGIN
        INSERT INTO emails_fts (rowid, subject, body)
        VALUES (new.id, new.subject, new.body);
    END;

    CREATE TRIGGER emails_ad AFTER DELETE ON emails BEGIN
        DELETE FROM emails_fts WHERE rowid=old.id;
    END;

    CREATE TRIGGER emails_au AFTER UPDATE ON emails BEGIN
        UPDATE emails_fts SET subject=new.subject, body=new.body WHERE rowid=old.id;
    END;
    """

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.executescript(SQL_CREATE_TABLES)
    conn.commit()

    print("Loading Enron emails from HF...")
    expected_features = Features(
        {
            "message_id": Value("string"),
            "subject": Value("string"),
            "from": Value("string"),
            "to": Sequence(Value("string")),
            "cc": Sequence(Value("string")),
            "bcc": Sequence(Value("string")),
            "date": Value("timestamp[us]"),
            "body": Value("string"),
            "file_name": Value("string"),
        }
    )

    dataset = load_dataset(
        EMAIL_DATASET_REPO_ID, features=expected_features, split="train"
    )
    print(f"Dataset contains {len(dataset)} emails")

    print("Populating SQLite DB (may take a few minutes)...")
    conn.execute("PRAGMA synchronous = OFF;")
    conn.execute("PRAGMA journal_mode = MEMORY;")
    conn.execute("BEGIN TRANSACTION;")

    record_count = 0
    skipped_count = 0
    duplicate_count = 0
    processed_emails = set()

    for email_data in tqdm(dataset, desc="Inserting emails"):
        message_id = email_data["message_id"]
        subject = email_data["subject"]
        from_address = email_data["from"]
        date_obj: datetime = email_data["date"]
        body = email_data["body"]
        file_name = email_data["file_name"]

        to_list = [str(a) for a in email_data["to"] if a]
        cc_list = [str(a) for a in email_data["cc"] if a]
        bcc_list = [str(a) for a in email_data["bcc"] if a]

        total_recipients = len(to_list) + len(cc_list) + len(bcc_list)

        if len(body) > 5000:
            skipped_count += 1
            continue
        if total_recipients > 30:
            skipped_count += 1
            continue

        email_key = (subject, body, from_address)
        if email_key in processed_emails:
            duplicate_count += 1
            continue
        processed_emails.add(email_key)

        date_str = date_obj.strftime("%Y-%m-%d %H:%M:%S")

        cursor.execute(
            """
            INSERT INTO emails (message_id, subject, from_address, date, body, file_name)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (message_id, subject, from_address, date_str, body, file_name),
        )

        recipient_data = []
        for addr in to_list:
            recipient_data.append((message_id, addr, "to"))
        for addr in cc_list:
            recipient_data.append((message_id, addr, "cc"))
        for addr in bcc_list:
            recipient_data.append((message_id, addr, "bcc"))

        if recipient_data:
            cursor.executemany(
                """
                INSERT INTO recipients (email_id, recipient_address, recipient_type)
                VALUES (?, ?, ?)
                """,
                recipient_data,
            )

        record_count += 1

    conn.commit()

    print("Creating FTS indexes and triggers...")
    cursor.executescript(SQL_CREATE_INDEXES_TRIGGERS)
    cursor.execute('INSERT INTO emails_fts(emails_fts) VALUES("rebuild")')
    conn.commit()

    print(f"DB ready with {record_count} emails; skipped {skipped_count}, duplicates {duplicate_count}.")
    return conn

In [8]:
def get_db_connection():
    global db_conn
    if db_conn is None:
        if os.path.exists(DB_PATH):
            print(f"Loading existing DB from {DB_PATH}")
            db_conn = sqlite3.connect(DB_PATH, check_same_thread=False)
        else:
            db_conn = create_email_database()
    return db_conn

def search_emails(
    inbox: str,
    keywords: List[str],
    from_addr: Optional[str] = None,
    to_addr: Optional[str] = None,
    sent_after: Optional[str] = None,
    sent_before: Optional[str] = None,
    max_results: int = 10,
) -> List[SearchResult]:
    conn = get_db_connection()
    cursor = conn.cursor()

    if not keywords:
        raise ValueError("No keywords provided for search.")
    if max_results > 40:
        raise ValueError("max_results must be <= 10.")

    where_clauses: List[str] = []
    params: List[str | int] = []

    # FTS query
    fts_query = " ".join(f' "{k.replace("\"", "\"\"")}" ' for k in keywords)
    where_clauses.append("fts.emails_fts MATCH ?")
    params.append(fts_query)

    # Inbox filter
    where_clauses.append(
        """
        (e.from_address = ? OR EXISTS (
            SELECT 1 FROM recipients r_inbox
            WHERE r_inbox.recipient_address = ? AND r_inbox.email_id = e.message_id
        ))
        """
    )
    params.extend([inbox, inbox])

    if from_addr:
        where_clauses.append("e.from_address = ?")
        params.append(from_addr)
    if to_addr:
        where_clauses.append(
            """
            EXISTS (
                SELECT 1 FROM recipients r_to
                WHERE r_to.recipient_address = ? AND r_to.email_id = e.message_id
            )
            """
        )
        params.append(to_addr)
    if sent_after:
        where_clauses.append("e.date >= ?")
        params.append(f"{sent_after} 00:00:00")
    if sent_before:
        where_clauses.append("e.date < ?")
        params.append(f"{sent_before} 00:00:00")

    sql = f"""
        SELECT
            e.message_id,
            snippet(emails_fts, -1, '<b>', '</b>', ' ... ', 15) as snippet
        FROM
            emails e JOIN emails_fts fts ON e.id = fts.rowid
        WHERE
            {" AND ".join(where_clauses)}
        ORDER BY
            e.date DESC
        LIMIT ?;
    """
    params.append(max_results)

    cursor.execute(sql, params)
    rows = cursor.fetchall()
    return [SearchResult(message_id=r[0], snippet=r[1]) for r in rows]

def read_email(message_id: str) -> Optional[Email]:
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT message_id, date, subject, from_address, body, file_name FROM emails WHERE message_id = ?",
        (message_id,),
    )
    row = cursor.fetchone()
    if not row:
        return None

    msg_id, date, subject, from_addr, body, file_name = row

    cursor.execute(
        "SELECT recipient_address, recipient_type FROM recipients WHERE email_id = ?",
        (message_id,),
    )
    rec_rows = cursor.fetchall()

    to_addresses, cc_addresses, bcc_addresses = [], [], []
    for addr, typ in rec_rows:
        t = typ.lower()
        if t == "to":
            to_addresses.append(addr)
        elif t == "cc":
            cc_addresses.append(addr)
        elif t == "bcc":
            bcc_addresses.append(addr)

    return Email(
        message_id=msg_id,
        date=date,
        subject=subject,
        from_address=from_addr,
        to_addresses=to_addresses,
        cc_addresses=cc_addresses,
        bcc_addresses=bcc_addresses,
        body=body,
        file_name=file_name,
    )

def load_scenarios(
    split: Literal["train", "test"] = "train",
    limit: Optional[int] = None,
    max_messages: Optional[int] = 1,
    shuffle: bool = False,
    seed: Optional[int] = None,
) -> List[Scenario]:
    print(f"Loading {split} scenarios from Hugging Face...")
    dataset: Dataset = load_dataset(SCENARIO_DATASET_REPO_ID, split=split)

    if max_messages is not None:
        dataset = dataset.filter(lambda x: len(x["message_ids"]) <= max_messages)

    if shuffle or seed is not None:
        if seed is not None:
            dataset = dataset.shuffle(seed=seed)
        else:
            dataset = dataset.shuffle()

    scenarios = [Scenario(**row, split=split) for row in dataset]

    if max_messages is not None:
        scenarios = [s for s in scenarios if len(s.message_ids) <= max_messages]

    if shuffle:
        if seed is not None:
            rng = random.Random(seed)
            rng.shuffle(scenarios)
        else:
            random.shuffle(scenarios)

    if limit is not None:
        scenarios = scenarios[:limit]

    print(f"Loaded {len(scenarios)} scenarios.")
    return scenarios

In [9]:
# Load 340 total samples: 150 for SFT, 150 for RLHF, 40 for validation
# We need non-overlapping datasets for proper training

# Load all training scenarios and split them
all_train_scenarios = load_scenarios("train", limit=300, max_messages=1, shuffle=True, seed=42)

# Split into SFT and RLHF datasets (non-overlapping)
sft_scenarios = all_train_scenarios[:150]  # First 150 for SFT
rlhf_scenarios = all_train_scenarios[150:300]  # Next 150 for RLHF

# Load validation scenarios
validation_scenarios = load_scenarios("test", limit=40, max_messages=1, shuffle=True, seed=42)

print(f"Loaded {len(sft_scenarios)} SFT scenarios")
print(f"Loaded {len(rlhf_scenarios)} RLHF scenarios")
print(f"Loaded {len(validation_scenarios)} validation scenarios")
print(f"Total: {len(sft_scenarios) + len(rlhf_scenarios) + len(validation_scenarios)} scenarios")
print("\
Datasets are non-overlapping:")
print(f"  SFT:        scenarios 0-149")
print(f"  RLHF:       scenarios 150-299")
print(f"  Validation: separate test set")
print("\
Sample SFT scenario:")
print(sft_scenarios[0])
print("\
Sample RLHF scenario:")
print(rlhf_scenarios[0])

Loading train scenarios from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/631 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/681k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/268k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1733 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4390 [00:00<?, ? examples/s]

Loaded 300 scenarios.
Loading test scenarios from Hugging Face...


Filter:   0%|          | 0/1733 [00:00<?, ? examples/s]

Loaded 40 scenarios.
Loaded 150 SFT scenarios
Loaded 150 RLHF scenarios
Loaded 40 validation scenarios
Total: 340 scenarios
Datasets are non-overlapping:
  SFT:        scenarios 0-149
  RLHF:       scenarios 150-299
  Validation: separate test set
Sample SFT scenario:
id=3296 question='Who can I contact for Power Operations when Sally is in London?' answer='Stacey White (x31870) and Leslie Reeves (x37962).' message_ids=['<6033065.1075856098960.JavaMail.evans@thyme>'] how_realistic=0.699999988079071 inbox_address='louise.kitchen@enron.com' query_date='2001-01-25' split='train'
Sample RLHF scenario:
id=3325 question='How will PG&E structure their hydro asset auction?' answer='The auction will be held in two stages: a Request for Qualifications and then a binding bid process. Assets will be auctioned by watershed or in bundles within a watershed, making 25 possible bid packages.' message_ids=['<5191831.1075842931935.JavaMail.evans@thyme>'] how_realistic=0.699999988079071 inbox_address='je

In [10]:
# -------------------------
# Tinker model + judge
# -------------------------
import tinker
from tinker import types, TensorData

# Explicit submodule imports
import tinker_cookbook.renderers as renderers
import tinker_cookbook.model_info as model_info
from tinker_cookbook.tokenizer_utils import get_tokenizer

import torch

In [26]:
# BASE_MODEL_NAME = "meta-llama/Llama-3.2-1B"  # adjust if needed
BASE_MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507" #instruct model

service_client = tinker.ServiceClient()

policy_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL_NAME,
    rank=32,
)

tokenizer = get_tokenizer(BASE_MODEL_NAME)
renderer_name = model_info.get_recommended_renderer_name(BASE_MODEL_NAME)
renderer = renderers.get_renderer(renderer_name, tokenizer)
print("Using renderer:", renderer_name)

JUDGE_MODEL_NAME = BASE_MODEL_NAME

judge_training_client = service_client.create_lora_training_client(
    base_model=JUDGE_MODEL_NAME,
    rank=4,
)

judge_sampling_path = judge_training_client.save_weights_for_sampler(
    name="email-judge-init",
).result().path

judge_sampling_client = service_client.create_sampling_client(
    model_path=judge_sampling_path,
)

judge_tokenizer = get_tokenizer(JUDGE_MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Using renderer: qwen3_instruct


In [11]:
class CorrectnessJudgeResponse(BaseModel):
    reasoning: str = Field(
        description="Short explanation of why the answer is correct or not."
    )
    accept: bool = Field(
        description="Whether the candidate answer should be accepted as correct."
    )

def build_json_judge_prompt(
    question: str,
    reference_answer: str,
    candidate_answer: str,
) -> str:
    return f"""You are a strict correctness judge for an email QA task.

You are given:
- A question about emails
- A reference (ground truth) answer
- A candidate answer generated by an AI assistant

Your job is to decide whether the candidate answer should be accepted as correct.
Consider whether it captures the same essential information as the reference answer,
even if the wording is different.

You MUST respond with a single JSON object matching exactly this schema:

{{
  "reasoning": "short explanation string",
  "accept": true or false
}}

Examples:
{{
  "reasoning": "Candidate mentions both Stacey White and Leslie Reeves with correct roles.",
  "accept": true
}}

{{
  "reasoning": "Candidate mentions only Stacey White, missing Leslie Reeves.",
  "accept": false
}}

Now judge the candidate answer.

Question:
{question}

Reference answer:
{reference_answer}

Candidate answer:
{candidate_answer}

Respond with ONLY the JSON object, and nothing else.
"""

In [12]:
def judge_candidate_with_tinker_json(
    question: str,
    reference_answer: str,
    candidate_answer: str,
) -> CorrectnessJudgeResponse:
    prompt = build_json_judge_prompt(question, reference_answer, candidate_answer)
    input_ids = judge_tokenizer.encode(prompt, add_special_tokens=False)
    model_input = types.ModelInput.from_ints(tokens=input_ids)

    judge_params = types.SamplingParams(
        max_tokens=128,
        temperature=0.0,
        stop=["\n\n"],
    )

    result = judge_sampling_client.sample(
        prompt=model_input,
        num_samples=1,
        sampling_params=judge_params,
    ).result()

    out_tokens = result.sequences[0].tokens
    raw_content = judge_tokenizer.decode(out_tokens).strip()

    try:
        start = raw_content.find("{")
        end = raw_content.rfind("}")
        if start != -1 and end != -1 and end > start:
            raw_content = raw_content[start : end + 1]

        return CorrectnessJudgeResponse.model_validate_json(raw_content)
    except Exception as e:
        return CorrectnessJudgeResponse(
            reasoning=f"Parse error: {e}\
Raw: {raw_content}",
            accept=False,
        )

In [13]:
# -------------------------
# Tool-calling environment
# -------------------------

import re

SYSTEM_PROMPT = """You are an email search agent.

You are given a question about the Enron email corpus and access to the following tools:

1. search_inbox
   - arguments: {"keywords": [string], "max_results": int}
   - Use this to search the inbox for relevant emails. Returns:
     [{"message_id": str, "snippet": str}, ...].

2. read_email
   - arguments: {"message_id": str}
   - Use this to read a specific email in full.

3. return_final_answer
   - arguments: {"answer": str, "reference_message_ids": [str]}
   - Use this when you are ready to give the final answer. You must include
     the list of message_ids that support your answer.

You MUST interact only by emitting JSON tool calls of the form:

{"tool": "search_inbox", "arguments": { ... }}
{"tool": "read_email", "arguments": { ... }}
{"tool": "return_final_answer", "answer": "...", "reference_message_ids": ["<id>", ...]}

Do NOT answer directly in natural language without calling return_final_answer.
"""

def format_messages(messages: list[dict]) -> str:
    lines: list[str] = []
    for m in messages:
        role = m["role"]
        content = m["content"].strip()
        if role == "system":
            lines.append(f"[SYSTEM]\
{content}\
")
        elif role == "user":
            lines.append(f"[USER]\
{content}\
")
        elif role == "assistant":
            lines.append(f"[ASSISTANT]\
{content}\
")
        elif role == "tool":
            name = m.get("name", "tool")
            lines.append(f"[TOOL:{name}]\
{content}\
")
    lines.append(
        "[ASSISTANT]\
"
        "Respond ONLY with a JSON object describing the next tool call.\
"
    )
    return "\
".join(lines)

def parse_tool_call(raw_text: str):
    raw_text = raw_text.strip()
    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    snippet = raw_text[start : end + 1]
    try:
        data = json.loads(snippet)
    except Exception:
        return None
    if not isinstance(data, dict):
        return None
    if "tool" not in data or not isinstance(data["tool"], str):
        return None
    return data

@dataclass
class RolloutSample:
    prompt_text: str
    answer_text: str
    answer_tokens: List[int]
    answer_logprobs: List[float]

def run_tool_rollout(
    sampling_client,
    tokenizer,
    scenario: Scenario,
    max_tool_steps: int = 5,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
) -> RolloutSample:
    question = scenario.question
    inbox = scenario.inbox_address

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Inbox: {inbox}\
"
                f"Query date: {scenario.query_date}\
"
                f"Question: {question}"
            ),
        },
    ]

    final_answer_text = None

    for step in range(max_tool_steps):
        prompt_text = format_messages(messages)
        prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
        model_input = types.ModelInput.from_ints(prompt_ids)

        sampling_params = types.SamplingParams(
            max_tokens=max_new_tokens,
            temperature=temperature,
        )

        result = sampling_client.sample(
            prompt=model_input,
            num_samples=1,
            sampling_params=sampling_params,
        ).result()

        seq = result.sequences[0]
        out_tokens = seq.tokens
        out_logprobs = seq.logprobs
        if out_logprobs is None:
            raise RuntimeError("Sampling result missing logprobs.")

        out_text = tokenizer.decode(out_tokens)
        messages.append({"role": "assistant", "content": out_text})
        tool_call = parse_tool_call(out_text)

        if tool_call is None:
            final_answer_text = out_text.strip()
            return RolloutSample(
                prompt_text=prompt_text,
                answer_text=final_answer_text,
                answer_tokens=out_tokens,
                answer_logprobs=out_logprobs,
            )

        tool = tool_call["tool"]

        if tool == "search_inbox":
            args = tool_call.get("arguments", {})
            keywords = args.get("keywords", [])
            if isinstance(keywords, str):
                keywords = [keywords]
            max_results = int(args.get("max_results", 5))
            if not keywords:
                keywords = [w for w in re.findall(r"\w+", question) if len(w) > 2][:4]

            results = search_emails(
                inbox=inbox,
                keywords=keywords,
                max_results=max_results,
            )
            tool_text = "\
".join(
                f"- {r.message_id}: {r.snippet}" for r in results
            ) or "[]"
            messages.append(
                {"role": "tool", "name": "search_inbox", "content": tool_text}
            )

        elif tool == "read_email":
            args = tool_call.get("arguments", {})
            msg_id = str(args.get("message_id", "")).strip()
            email_obj = read_email(msg_id)
            if email_obj is None:
                tool_text = f"Email {msg_id} not found."
            else:
                tool_text = (
                    f"message_id: {email_obj.message_id}\
"
                    f"date: {email_obj.date}\
"
                    f"from: {email_obj.from_address}\
"
                    f"subject: {email_obj.subject}\
\
"
                    f"body:\
{email_obj.body or ''}"
                )
            messages.append(
                {"role": "tool", "name": "read_email", "content": tool_text}
            )

        elif tool == "return_final_answer":
            answer_text = tool_call.get("answer", "").strip()
            final_answer_text = answer_text
            return RolloutSample(
                prompt_text=prompt_text,
                answer_text=answer_text,
                answer_tokens=out_tokens,
                answer_logprobs=out_logprobs,
            )

        else:
            final_answer_text = out_text.strip()
            return RolloutSample(
                prompt_text=prompt_text,
                answer_text=final_answer_text,
                answer_tokens=out_tokens,
                answer_logprobs=out_logprobs,
            )

    if final_answer_text is None:
        final_answer_text = "I could not determine an answer from the available emails."
    return RolloutSample(
        prompt_text=prompt_text,
        answer_text=final_answer_text,
        answer_tokens=out_tokens,
        answer_logprobs=out_logprobs or [0.0] * len(out_tokens),
    )

def build_datums_from_rollouts(
    tokenizer,
    rollouts: list[RolloutSample],
    advantages: list[float],
):
    datums: list[types.Datum] = []

    for r, adv in zip(rollouts, advantages):
        prompt_tokens = tokenizer.encode(r.prompt_text, add_special_tokens=False)
        answer_tokens = list(map(int, r.answer_tokens))
        answer_logprobs = list(map(float, r.answer_logprobs))

        full_tokens = prompt_tokens + answer_tokens
        if len(full_tokens) < 2:
            continue

        input_tokens = full_tokens[:-1]
        target_tokens = full_tokens[1:]

        L = len(prompt_tokens)

        zeros_prefix = [0.0] * max(0, L - 1)
        logprobs_full = zeros_prefix + answer_logprobs
        logprobs_full = logprobs_full[: len(input_tokens)]

        adv_prefix = [0.0] * max(0, L - 1)
        adv_suffix = [adv] * max(0, len(input_tokens) - len(adv_prefix))
        advantages_full = adv_prefix + adv_suffix
        advantages_full = advantages_full[: len(input_tokens)]

        assert len(input_tokens) == len(target_tokens) == len(logprobs_full) == len(
            advantages_full
        )

        datum = types.Datum(
            model_input=types.ModelInput.from_ints(tokens=input_tokens),
            loss_fn_inputs={
                "target_tokens": TensorData.from_torch(
                    torch.tensor(target_tokens, dtype=torch.long)
                ),
                "logprobs": TensorData.from_torch(
                    torch.tensor(logprobs_full, dtype=torch.float32)
                ),
                "advantages": TensorData.from_torch(
                    torch.tensor(advantages_full, dtype=torch.float32)
                ),
            },
        )
        datums.append(datum)

    return datums

In [14]:
## 🔥 SFT Warmup: Curate 150 High-Quality Samples from Database
"""
Before running RLHF, we'll do supervised fine-tuning on 150 curated examples.
This extensive SFT warmup (using 93.75% of our training data) provides a strong foundation
before reinforcement learning refinement.
"""

"\nBefore running RLHF, we'll do supervised fine-tuning on 150 curated examples.\nThis extensive SFT warmup (using 93.75% of our training data) provides a strong foundation\nbefore reinforcement learning refinement.\n"

In [15]:
# ============================================================
# SFT WARMUP: Curate high-quality samples from database
# ============================================================

def create_sft_demonstration(
    scenario: Scenario,
    include_tool_steps: bool = True
) -> Optional[dict]:
    """
    Create a supervised training example by demonstrating the correct
    tool usage for a scenario.
    """
    question = scenario.question
    inbox = scenario.inbox_address
    answer = scenario.answer
    message_ids = scenario.message_ids

    # Build the user prompt
    user_prompt = f"""Inbox: {inbox}
Query date: {scenario.query_date}
Question: {question}"""

    # Build the demonstration response
    demo_steps = []

    if include_tool_steps and message_ids:
        # Extract keywords from question for search
        keywords = [w for w in re.findall(r"\w+", question) if len(w) > 2][:5]

        # Step 1: Search emails
        demo_steps.append(
            json.dumps({
                "tool": "search_inbox",
                "arguments": {
                    "keywords": keywords,
                    "max_results": min(10, len(message_ids) * 2)
                }
            })
        )

        # Step 2: Read relevant emails
        for msg_id in message_ids[:2]:  # Read at most 2 emails
            demo_steps.append(
                json.dumps({
                    "tool": "read_email",
                    "arguments": {"message_id": msg_id}
                })
            )

    # Final step: Return answer
    demo_steps.append(
        json.dumps({
            "tool": "return_final_answer",
            "answer": answer,
            "reference_message_ids": message_ids
        })
    )

    # Combine into full response
    assistant_response = "\
".join(demo_steps)

    return {
        "system": SYSTEM_PROMPT,
        "user": user_prompt,
        "assistant": assistant_response,
        "scenario_id": scenario.id
    }


def curate_sft_dataset(
    scenarios: List[Scenario],
    num_samples: int = 20,
    quality_threshold: float = 0.7
) -> List[dict]:
    """
    Curate high-quality SFT training samples from scenarios.
    """
    print(f"Curating {num_samples} SFT samples from {len(scenarios)} scenarios...")

    # Filter high-quality scenarios
    high_quality = [s for s in scenarios if s.how_realistic >= quality_threshold]
    print(f"Found {len(high_quality)} high-quality scenarios (realistic >= {quality_threshold})")

    # Select samples
    selected = random.sample(
        high_quality,
        min(num_samples, len(high_quality))
    )

    # Create demonstrations
    sft_samples = []
    for scenario in tqdm(selected, desc="Creating SFT demonstrations"):
        demo = create_sft_demonstration(scenario, include_tool_steps=True)
        if demo:
            sft_samples.append(demo)

    print(f"Created {len(sft_samples)} SFT training samples")
    return sft_samples




In [16]:
def build_sft_datums(tokenizer, sft_samples):
    """
    Build datums using importance_sampling (which works in RLHF).
    For SFT, we use uniform advantages and zero logprobs.
    """
    datums = []

    for sample in sft_samples:
        # Create full conversation
        full_text = f"{sample['system']}\n\nUser: {sample['user']}\n\nAssistant: {sample['assistant']}"

        # Tokenize
        tokens = tokenizer.encode(full_text, add_special_tokens=False)

        if len(tokens) < 2:
            continue

        input_tokens = tokens[:-1]
        target_tokens = tokens[1:]

        # For SFT with importance_sampling:
        # - advantages: all 1.0 (no preference, just supervised learning)
        # - logprobs: all 0.0 (will be replaced by model's actual logprobs)
        advantages = [1.0] * len(input_tokens)
        logprobs = [0.0] * len(input_tokens)

        datum = types.Datum(
            model_input=types.ModelInput.from_ints(tokens=input_tokens),
            loss_fn_inputs={
                "target_tokens": TensorData.from_torch(
                    torch.tensor(target_tokens, dtype=torch.long)
                ),
                "advantages": TensorData.from_torch(
                    torch.tensor(advantages, dtype=torch.float32)
                ),
                "logprobs": TensorData.from_torch(
                    torch.tensor(logprobs, dtype=torch.float32)
                ),
            },
        )
        datums.append(datum)

    return datums

def run_sft_warmup(
    training_client,
    tokenizer,
    scenarios,
    num_samples=150,
    num_epochs=3,
    learning_rate=2e-5,
):
    """
    Run SFT using importance_sampling loss (which we know works).
    """
    import uuid
    from datetime import datetime

    # Generate unique ID for this SFT run to avoid checkpoint conflicts
    sft_run_id = uuid.uuid4().hex[:8]

    print("\n" + "="*50)
    print("STARTING SFT WARMUP")
    print(f"Run ID: {sft_run_id}")
    print("="*50)

    # Curate samples
    sft_samples = curate_sft_dataset(
        scenarios=scenarios,
        num_samples=num_samples,
        quality_threshold=0.5  # Lower threshold
    )

    # Build datums with importance_sampling format
    datums = build_sft_datums(tokenizer, sft_samples)
    print(f"\nTotal SFT datums: {len(datums)}")

    if len(datums) == 0:
        print("ERROR: No datums created")
        return None

    # Training parameters
    adam_params = types.AdamParams(
        learning_rate=learning_rate,
        beta1=0.9,
        beta2=0.999,
        eps=1e-8,
    )

    # Run training epochs
    for epoch in range(num_epochs):
        print(f"\n--- SFT Epoch {epoch + 1}/{num_epochs} ---")

        epoch_datums = random.sample(datums, len(datums))
        batch_size = 4

        for i in tqdm(range(0, len(epoch_datums), batch_size), desc=f"Epoch {epoch+1}"):
            batch = epoch_datums[i:i+batch_size]

            try:
                # Use importance_sampling instead of cross_entropy
                fwd_bwd_fut = training_client.forward_backward(
                    batch,
                    loss_fn="importance_sampling",  # Changed from cross_entropy
                )

                optim_fut = training_client.optim_step(adam_params)

                _ = fwd_bwd_fut.result()
                _ = optim_fut.result()

            except Exception as e:
                print(f"Warning: Batch {i//batch_size} failed: {e}")
                continue

    print("\n" + "="*50)
    print("SFT WARMUP COMPLETE")
    print("="*50)

    # Save checkpoint with unique name to avoid conflicts
    checkpoint_name = f"email-agent-sft-warmup-{sft_run_id}"
    sft_path = training_client.save_weights_for_sampler(
        name=checkpoint_name
    ).result().path

    print(f"SFT checkpoint saved as: {checkpoint_name}")
    print(f"SFT checkpoint path: {sft_path}")
    return sft_path

In [ ]:
# Run SFT warmup with dedicated SFT dataset
# Using 150 scenarios that are SEPARATE from RLHF scenarios
sft_checkpoint_path = run_sft_warmup(
    training_client=policy_training_client,
    tokenizer=tokenizer,
    scenarios=sft_scenarios,  # 150 dedicated SFT scenarios
    num_samples=150,  # Use all 150 SFT scenarios
    num_epochs=3,     # 3 epochs of supervised training
    learning_rate=2e-5,
)

print(f"\
SFT warmup complete! Model trained on 150 dedicated SFT examples.")
print(f"These scenarios will NOT be used in RLHF training.")
print(f"Checkpoint: {sft_checkpoint_path}")


STARTING SFT WARMUP
Run ID: 5b2329a2
Curating 150 SFT samples from 150 scenarios...
Found 150 high-quality scenarios (realistic >= 0.5)


Creating SFT demonstrations:   0%|          | 0/150 [00:00<?, ?it/s]

Created 150 SFT training samples

Total SFT datums: 150

--- SFT Epoch 1/3 ---


Epoch 1:   0%|          | 0/38 [00:00<?, ?it/s]


--- SFT Epoch 2/3 ---


Epoch 2:   0%|          | 0/38 [00:00<?, ?it/s]


--- SFT Epoch 3/3 ---


Epoch 3:   0%|          | 0/38 [00:00<?, ?it/s]


SFT WARMUP COMPLETE
SFT checkpoint saved as: email-agent-sft-warmup-5b2329a2
SFT checkpoint path: tinker://f87fddc8-75e4-50cd-a0ad-f603ab570aac:train:0/sampler_weights/email-agent-sft-warmup-5b2329a2
SFT warmup complete! Model trained on 150 dedicated SFT examples.
These scenarios will NOT be used in RLHF training.
Checkpoint: tinker://f87fddc8-75e4-50cd-a0ad-f603ab570aac:train:0/sampler_weights/email-agent-sft-warmup-5b2329a2


In [ ]:
## 🚀 RLHF Training
"""
Now we run RLHF training, starting from the SFT-warmed model.
"""

'\nNow we run RLHF training, starting from the SFT-warmed model.\n'

In [ ]:
from tqdm.auto import trange

# Parameters scaled for 150 RLHF scenarios (separate from SFT)
GROUP_SIZE = 8      # Number of rollouts per scenario for variance estimation
BATCH_SIZE = 8      # Number of scenarios per training step
MAX_STEPS = 20      # Total RLHF training steps

sampling_params = types.SamplingParams(
    max_tokens=256,
    temperature=0.7,
)

adam_params = types.AdamParams(
    learning_rate=1e-5,
    beta1=0.9,
    beta2=0.95,
    eps=1e-8,
)

n_rlhf = len(rlhf_scenarios)
print(f"RLHF training scenarios: {n_rlhf} (separate from SFT)")
print(f"Scenarios per step: {BATCH_SIZE}")
print(f"Total training steps: {MAX_STEPS}")
print(f"Total scenarios to process: {BATCH_SIZE * MAX_STEPS}")

RLHF training scenarios: 150 (separate from SFT)
Scenarios per step: 8
Total training steps: 20
Total scenarios to process: 160


In [17]:
import uuid
RUN_ID = uuid.uuid4().hex[:8]  # new each notebook run


def train_email_agent(debug: bool = True):
    step = 0
    while step < MAX_STEPS:
        print(f"\
================= STEP {step} =================")
        ckpt_name = f"email-agent-{RUN_ID}-step-{step:06d}"
        sampler_path = policy_training_client.save_weights_for_sampler(
            name=ckpt_name
        ).result().path

        sampling_client = service_client.create_sampling_client(
            model_path=sampler_path,
        )

        batch_scenarios = random.sample(
            rlhf_scenarios,
            min(BATCH_SIZE, len(rlhf_scenarios)),
        )

        if debug:
            print(f"[step {step}] batch_scenarios: {[s.id for s in batch_scenarios]}")

        all_datums: list[types.Datum] = []
        batch_rewards: list[float] = []
        num_groups = 0
        num_groups_used = 0
        num_zero_adv_groups = 0

        for scenario in batch_scenarios:
            num_groups += 1
            if debug:
                print(f"\
  Scenario {scenario.id}: {scenario.question[:80]!r}")

            # ---- rollouts for this scenario ----
            rollouts: list[RolloutSample] = []
            for g in range(GROUP_SIZE):
                r = run_tool_rollout(
                    sampling_client=sampling_client,
                    tokenizer=tokenizer,
                    scenario=scenario,
                    max_tool_steps=5,
                    max_new_tokens=128,
                    temperature=0.7,
                )
                rollouts.append(r)

            # ---- judge rewards ----
            rewards: list[float] = []
            judge_responses: list[CorrectnessJudgeResponse] = []

            for idx, r in enumerate(rollouts):
                judge_resp = judge_candidate_with_tinker_json(
                    question=scenario.question,
                    reference_answer=scenario.answer,
                    candidate_answer=r.answer_text,
                )
                reward = 1.0 if judge_resp.accept else 0.0
                rewards.append(reward)
                judge_responses.append(judge_resp)

                if debug:
                    print(f"    rollout {idx}: reward={reward}, "
                          f"accept={judge_resp.accept}, "
                          f"ans={r.answer_text[:70]!r}")

            if not rewards:
                if debug:
                    print("    (no rewards, skipping scenario)")
                continue

            mean_reward = sum(rewards) / len(rewards)
            advantages = [rv - mean_reward for rv in rewards]

            if debug:
                print(f"    group rewards:     {rewards}")
                print(f"    group mean_reward: {mean_reward:.3f}")
                print(f"    group advantages:  {[round(a, 3) for a in advantages]}")

            # All rollouts same reward => advantages all zero => skip this group
            if all(abs(a) < 1e-6 for a in advantages):
                num_zero_adv_groups += 1
                if debug:
                    print("    -> all advantages ~0, skipping this group")
                continue

            num_groups_used += 1
            batch_rewards.append(mean_reward)
            datums = build_datums_from_rollouts(tokenizer, rollouts, advantages)
            all_datums.extend(datums)

        # ---- summary for this step ----
        if not all_datums:
            print(
                f"[step {step}] no datums (all zero-advantage); "
                f"groups={num_groups}, zero-adv-groups={num_zero_adv_groups}"
            )
            step += 1
            continue

        if debug:
            print(
                f"[step {step}] forward/backward on {len(all_datums)} datums, "
                f"used_groups={num_groups_used}/{num_groups}, "
                f"zero_adv_groups={num_zero_adv_groups}"
            )

        # ---- GRPO / importance_sampling update ----
        fwd_bwd_fut = policy_training_client.forward_backward(
            all_datums,
            loss_fn="importance_sampling",
        )
        optim_fut = policy_training_client.optim_step(adam_params)
        _ = fwd_bwd_fut.result()
        _ = optim_fut.result()

        mean_batch_reward = (
            sum(batch_rewards) / len(batch_rewards) if batch_rewards else 0.0
        )
        print(
            f"[step {step}] datums={len(all_datums)}, "
            f"mean_group_reward={mean_batch_reward:.3f}, "
            f"used_groups={num_groups_used}/{num_groups}"
        )

        step += 1

def build_final_sampler():
    ckpt_name = f"email-agent-{RUN_ID}-final"
    sampler_path = policy_training_client.save_weights_for_sampler(
        name=ckpt_name
    ).result().path
    return service_client.create_sampling_client(model_path=sampler_path)

In [ ]:
# Configure training parameters for 150 RLHF scenarios
GROUP_SIZE = 8      # Increased for better variance estimation
BATCH_SIZE = 8      # Increased to process more scenarios per step
MAX_STEPS  = 20     # Training steps for RLHF dataset

# Run the full training pipeline
print(f"\
🎯 Starting RLHF training on {len(rlhf_scenarios)} dedicated RLHF scenarios...")
print(f"   (These are DIFFERENT from the {len(sft_scenarios)} SFT scenarios)\
")
train_email_agent(debug=True)

🎯 Starting RLHF training on 150 dedicated RLHF scenarios...
   (These are DIFFERENT from the 150 SFT scenarios)
================= STEP 0 =================
[step 0] batch_scenarios: [2691, 4983, 1093, 3607, 1867, 5879, 3632, 441]
  Scenario 2691: 'Which countries are currently on the boycott list that I need to consider for sy'
    rollout 0: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["boycott", "list",'
    rollout 1: reward=0.0, accept=False, ans='Do NOT include explanations or additional text.\n\n{"tool": "search_inbo'
    rollout 2: reward=0.0, accept=False, ans='Do NOT include any other text or explanation.\n\n{"tool": "search_inbox"'
    rollout 3: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["boycott", "list",'
    rollout 4: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["boycott", "list",'
    rollout 5: reward=0.0, accept=False, ans='{"tool": "search_inbox", "argum

README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/185M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/141M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/517401 [00:00<?, ? examples/s]

Dataset contains 517401 emails
Populating SQLite DB (may take a few minutes)...


Inserting emails:   0%|          | 0/517401 [00:00<?, ?it/s]

Creating FTS indexes and triggers...
DB ready with 225965 emails; skipped 57734, duplicates 233702.
    rollout 0: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["March", "7", "wor'
    rollout 1: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["March", "7", "wor'
    rollout 2: reward=0.0, accept=False, ans='Do NOT include explanations or commentary.\n\n{"tool": "search_inbox", "'
    rollout 3: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["March", "7", "wor'
    rollout 4: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["March", "7", "wor'
    rollout 5: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["March", "7", "wor'
    rollout 6: reward=0.0, accept=False, ans='{"tool": "search_inbox", "arguments": {"keywords": ["March", "7", "wor'
    rollout 7: reward=0.0, accept=False, ans='Do NOT include explanation

In [ ]:
def demo_inference():
    final_sampling_client = build_final_sampler()
    test_scenario = validation_scenarios[0]
    print("=== Test scenario ===")
    print("ID:", test_scenario.id)
    print("Question:", test_scenario.question)
    print("Reference answer:", test_scenario.answer)
    print("Expected message_ids:", test_scenario.message_ids)
    print()

    rollout = run_tool_rollout(
        sampling_client=final_sampling_client,
        tokenizer=tokenizer,
        scenario=test_scenario,
        max_tool_steps=5,
        max_new_tokens=128,
        temperature=0.0,
    )
    print("Agent final answer:", rollout.answer_text)

    judge_resp = judge_candidate_with_tinker_json(
        question=test_scenario.question,
        reference_answer=test_scenario.answer,
        candidate_answer=rollout.answer_text,
    )
    print("\
Judge reasoning:", judge_resp.reasoning)
    print("Judge accept:", judge_resp.accept)

# Test the final model
demo_inference()

=== Test scenario ===
ID: 1644
Question: What is the name of PentaSafe's new security software and when is it expected to be available?
Reference answer: VigilEnt Security Agent, expected to be available to the general market in the first quarter of 2001.
Expected message_ids: ['<30025152.1075857975007.JavaMail.evans@thyme>']

Agent final answer: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"keywords": ["PentaSafe", "security", "software", "available"], "max_results": 2}}{"tool": "read_email", "arguments": {"message_id": "12345"}}{"tool": "return_final_answer", "answer": "The name of PentaSafe's new security software is SentinelGuard, and it is expected to be available in Q2 2001.", "reference_message_ids": ["12345"]}<|endoftext|>
Judge reasoning: Candidate answer provides a different software name (SentinelGuard vs. VigilEnt Security Agent) and a different release quarter (Q2 2001 vs. first quarter of 2001), which contradicts the reference answer.
Jud

In [ ]:
## 📊 Complete Training & Evaluation Pipeline Summary
"""
This enhanced notebook implements a comprehensive email search agent with:

### **1. Dataset (340 Scenarios Total - NO OVERLAP)**:
   - 150 SFT scenarios (supervised fine-tuning only)
   - 150 RLHF scenarios (reinforcement learning only)
   - 40 validation scenarios (evaluation)
   - **Key**: SFT and RLHF use completely different scenarios

### **2. Two-Stage Training with Separate Data**:
   - **Stage 1 - SFT Warmup**:
     - 150 dedicated scenarios (never seen in RLHF)
     - 3 epochs = 450 total training passes
     - Builds strong foundation with supervised learning
   - **Stage 2 - RLHF Training**:
     - 150 different scenarios (never seen in SFT)
     - 20 steps × batch size 8 = 160 scenario samples
     - Refines model with judge-based rewards

### **3. Why Separate Datasets Matter**:
   - **No overfitting**: Model doesn't see same data twice
   - **Better generalization**: Tests true learning, not memorization
   - **Clean evaluation**: RLHF performance on unseen scenarios
   - **Proper train/test split**: Each phase has fresh data

### **4. Comprehensive Evaluation**:
   - Full validation set (40 scenarios)
   - Multiple rollouts per scenario
   - Success/failure analysis
   - Baseline comparison

The separation ensures the model learns general patterns in SFT,
then applies and refines them on new scenarios in RLHF,
leading to better generalization and true performance gains.
"""

"\nThis enhanced notebook implements a comprehensive email search agent with:\n\n### **1. Dataset (340 Scenarios Total - NO OVERLAP)**:\n   - 150 SFT scenarios (supervised fine-tuning only)\n   - 150 RLHF scenarios (reinforcement learning only)\n   - 40 validation scenarios (evaluation)\n   - **Key**: SFT and RLHF use completely different scenarios\n\n### **2. Two-Stage Training with Separate Data**:\n   - **Stage 1 - SFT Warmup**:\n     - 150 dedicated scenarios (never seen in RLHF)\n     - 3 epochs = 450 total training passes\n     - Builds strong foundation with supervised learning\n   - **Stage 2 - RLHF Training**:\n     - 150 different scenarios (never seen in SFT)\n     - 20 steps × batch size 8 = 160 scenario samples\n     - Refines model with judge-based rewards\n\n### **3. Why Separate Datasets Matter**:\n   - **No overfitting**: Model doesn't see same data twice\n   - **Better generalization**: Tests true learning, not memorization\n   - **Clean evaluation**: RLHF performance

In [37]:
# ============================================================
# EVALUATION FUNCTIONS
# ============================================================


def build_sampler_from_checkpoint(ckpt_name, BASE_MODEL_NAME="Qwen/Qwen3-30B-A3B-Instruct-2507"):


    if ckpt_name == BASE_MODEL_NAME:


      # BASE_MODEL_NAME = "meta-llama/Llama-3.2-1B"  # adjust if needed
      BASE_MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507" #instruct model

      service_client = tinker.ServiceClient()

      policy_training_client = service_client.create_lora_training_client(
          base_model=BASE_MODEL_NAME,
          rank=32,
      )

      tokenizer = get_tokenizer(BASE_MODEL_NAME)
      renderer_name = model_info.get_recommended_renderer_name(BASE_MODEL_NAME)
      renderer = renderers.get_renderer(renderer_name, tokenizer)
      print("Using renderer:", renderer_name)

      base_training_client = service_client.create_lora_training_client(
          base_model=BASE_MODEL_NAME,
          rank=4,
      )

      base_sampling_path = base_training_client.save_weights_for_sampler(
          name="base-model-init",
      ).result().path

      base_sampling_client = service_client.create_sampling_client(
          model_path=base_sampling_path,
      )

      #base_tokenizer = get_tokenizer(JUDGE_MODEL_NAME)


      return base_sampling_client

    else:
      service_client = tinker.ServiceClient()

      sampling_client = service_client.create_sampling_client(
          model_path=ckpt_name,
      )
      return sampling_client



def evaluate_agent_on_scenarios(
    sampling_client,
    tokenizer,
    scenarios,
    num_rollouts_per_scenario: int = 3,
    verbose: bool = False,
    max_scenarios: Optional[int] = None,
):
    """
    Comprehensive evaluation of the agent on a set of scenarios.

    Args:
        sampling_client: The Tinker sampling client to use
        tokenizer: Tokenizer for the model
        scenarios: List of scenarios to evaluate on
        num_rollouts_per_scenario: How many attempts per scenario (best is kept)
        verbose: Whether to print detailed output
        max_scenarios: Limit evaluation to first N scenarios (for testing)

    Returns:
        Dictionary with evaluation metrics and detailed records
    """
    from datetime import datetime

    eval_scenarios = scenarios[:max_scenarios] if max_scenarios else scenarios

    total = 0
    accepted = 0
    all_records = []

    # Track timing
    start_time = datetime.now()

    print(f"\n{'='*60}")
    print(f"EVALUATING ON {len(eval_scenarios)} SCENARIOS")
    print(f"{'='*60}")

    for s_idx, scenario in enumerate(tqdm(eval_scenarios, desc="Evaluating")):
        if verbose:
            print(f"\n=== Scenario {scenario.id} ({s_idx+1}/{len(eval_scenarios)}) ===")
            print(f"Q: {scenario.question[:100]}...")
            print(f"Ref: {scenario.answer[:100]}...")

        best_reward = -1.0
        best_answer = None
        best_judge = None
        all_attempts = []

        # Multiple rollouts per scenario
        for r_idx in range(num_rollouts_per_scenario):
            try:
                # Run the agent
                rollout = run_tool_rollout(
                    sampling_client=sampling_client,
                    tokenizer=tokenizer,
                    scenario=scenario,
                    max_tool_steps=5,
                    max_new_tokens=128,
                    temperature=0.1 if r_idx == 0 else 0.5,  # First rollout deterministic
                )

                # Judge the answer
                judge_resp = judge_candidate_with_tinker_json(
                    question=scenario.question,
                    reference_answer=scenario.answer,
                    candidate_answer=rollout.answer_text,
                )

                reward = 1.0 if judge_resp.accept else 0.0

                all_attempts.append({
                    "rollout_idx": r_idx,
                    "answer": rollout.answer_text,
                    "accepted": judge_resp.accept,
                    "reasoning": judge_resp.reasoning,
                })

                if verbose and num_rollouts_per_scenario > 1:
                    print(f"  Rollout {r_idx}: {'✓' if judge_resp.accept else '✗'}")

                # Keep best rollout
                if reward > best_reward:
                    best_reward = reward
                    best_answer = rollout.answer_text
                    best_judge = judge_resp

            except Exception as e:
                if verbose:
                    print(f"  Rollout {r_idx} failed: {e}")
                continue

        # Record results
        total += 1
        if best_reward > 0.0:
            accepted += 1

        record = {
            "scenario_id": scenario.id,
            "question": scenario.question,
            "reference_answer": scenario.answer,
            "expected_message_ids": scenario.message_ids,
            "best_answer": best_answer,
            "accepted": bool(best_reward > 0.0),
            "judge_reasoning": best_judge.reasoning if best_judge else "No valid rollouts",
            "num_attempts": len(all_attempts),
            "all_attempts": all_attempts,
        }
        all_records.append(record)

        if verbose:
            status = "✅ ACCEPTED" if best_reward > 0.0 else "❌ REJECTED"
            print(f"  Result: {status}")
            if best_answer:
                print(f"  Best answer: {best_answer[:150]}...")

    # Calculate metrics
    elapsed_time = (datetime.now() - start_time).total_seconds()
    accuracy = accepted / total if total > 0 else 0.0

    # Categorize failures
    failures = [r for r in all_records if not r["accepted"]]
    failure_categories = {
        "no_answer": 0,
        "wrong_answer": 0,
        "partial_answer": 0,
    }

    for failure in failures:
        if failure["best_answer"] is None:
            failure_categories["no_answer"] += 1
        elif "could not determine" in failure["best_answer"].lower():
            failure_categories["no_answer"] += 1
        elif any(word in failure["judge_reasoning"].lower() for word in ["partial", "incomplete", "missing"]):
            failure_categories["partial_answer"] += 1
        else:
            failure_categories["wrong_answer"] += 1

    return {
        "accuracy": accuracy,
        "num_scenarios": total,
        "num_accepted": accepted,
        "num_rejected": total - accepted,
        "records": all_records,
        "failure_categories": failure_categories,
        "elapsed_time_seconds": elapsed_time,
        "avg_time_per_scenario": elapsed_time / total if total > 0 else 0,
    }


def compare_models(
    baseline_client,
    trained_client,
    tokenizer,
    scenarios,
    num_rollouts: int = 2,
    verbose: bool = False,
):
    """
    Compare baseline model vs trained model performance.

    Args:
        baseline_client: Sampling client for baseline model
        trained_client: Sampling client for trained model
        tokenizer: Tokenizer
        scenarios: Scenarios to evaluate on
        num_rollouts: Rollouts per scenario
        verbose: Whether to print details

    Returns:
        Comparison dictionary with metrics
    """
    print("\n" + "="*70)
    print("MODEL COMPARISON: Baseline vs Trained")
    print("="*70)

    # Evaluate baseline
    print("\n📊 Evaluating Baseline Model...")
    baseline_results = evaluate_agent_on_scenarios(
        sampling_client=baseline_client,
        tokenizer=tokenizer,
        scenarios=scenarios,
        num_rollouts_per_scenario=num_rollouts,
        verbose=verbose,
    )

    # Evaluate trained
    print("\n📊 Evaluating Trained Model...")
    trained_results = evaluate_agent_on_scenarios(
        sampling_client=trained_client,
        tokenizer=tokenizer,
        scenarios=scenarios,
        num_rollouts_per_scenario=num_rollouts,
        verbose=verbose,
    )

    # Calculate improvements
    accuracy_improvement = trained_results["accuracy"] - baseline_results["accuracy"]
    accept_improvement = trained_results["num_accepted"] - baseline_results["num_accepted"]

    # Find scenarios that improved
    improved_scenarios = []
    degraded_scenarios = []

    for i in range(len(scenarios)):
        baseline_record = baseline_results["records"][i]
        trained_record = trained_results["records"][i]

        if not baseline_record["accepted"] and trained_record["accepted"]:
            improved_scenarios.append(trained_record["scenario_id"])
        elif baseline_record["accepted"] and not trained_record["accepted"]:
            degraded_scenarios.append(trained_record["scenario_id"])

    return {
        "baseline": baseline_results,
        "trained": trained_results,
        "accuracy_improvement": accuracy_improvement,
        "accept_improvement": accept_improvement,
        "improved_scenarios": improved_scenarios,
        "degraded_scenarios": degraded_scenarios,
        "comparison_summary": {
            "baseline_accuracy": baseline_results["accuracy"],
            "trained_accuracy": trained_results["accuracy"],
            "improvement_percentage": accuracy_improvement * 100,
            "baseline_accepted": baseline_results["num_accepted"],
            "trained_accepted": trained_results["num_accepted"],
            "scenarios_improved": len(improved_scenarios),
            "scenarios_degraded": len(degraded_scenarios),
        }
    }


def print_evaluation_report(eval_results, model_name="Model"):
    """
    Print a formatted evaluation report.
    """
    print("\n" + "="*70)
    print(f"📈 {model_name} EVALUATION REPORT")
    print("="*70)

    # Basic metrics
    print(f"\n📊 Overall Performance:")
    print(f"  Accuracy:        {eval_results['accuracy']:.2%}")
    print(f"  Accepted:        {eval_results['num_accepted']}/{eval_results['num_scenarios']}")
    print(f"  Rejected:        {eval_results['num_rejected']}/{eval_results['num_scenarios']}")

    # Failure analysis
    if eval_results["failure_categories"]:
        print(f"\n❌ Failure Breakdown:")
        for category, count in eval_results["failure_categories"].items():
            if count > 0:
                pct = (count / eval_results["num_rejected"]) * 100 if eval_results["num_rejected"] > 0 else 0
                print(f"  {category.replace('_', ' ').title()}: {count} ({pct:.1f}%)")

    # Timing
    print(f"\n⏱️ Performance:")
    print(f"  Total time:      {eval_results['elapsed_time_seconds']:.1f}s")
    print(f"  Avg per scenario: {eval_results['avg_time_per_scenario']:.2f}s")

    # Sample successes and failures
    successes = [r for r in eval_results["records"] if r["accepted"]]
    failures = [r for r in eval_results["records"] if not r["accepted"]]

    if successes:
        print(f"\n✅ Sample Successes:")
        for record in successes[:3]:
            print(f"\n  Q: {record['question'][:80]}...")
            print(f"  A: {record['best_answer'][:80]}...")

    if failures:
        print(f"\n❌ Sample Failures:")
        for record in failures[:3]:
            print(f"\n  Q: {record['question'][:80]}...")
            print(f"  Expected: {record['reference_answer'][:80]}...")
            print(f"  Got: {(record['best_answer'] or 'No answer')[:80]}...")
            print(f"  Reason: {record['judge_reasoning'][:100]}...")


def print_comparison_report(comparison_results):
    """
    Print a formatted comparison report between baseline and trained models.
    """
    summary = comparison_results["comparison_summary"]

    print("\n" + "="*70)
    print("🔄 MODEL COMPARISON REPORT")
    print("="*70)

    print("\n📊 Accuracy Comparison:")
    print(f"  Baseline:        {summary['baseline_accuracy']:.2%}")
    print(f"  Trained:         {summary['trained_accuracy']:.2%}")
    print(f"  Improvement:     {summary['improvement_percentage']:+.1f}%")

    print("\n📈 Acceptance Comparison:")
    print(f"  Baseline:        {summary['baseline_accepted']} accepted")
    print(f"  Trained:         {summary['trained_accepted']} accepted")
    print(f"  Net change:      {summary['trained_accepted'] - summary['baseline_accepted']:+d}")

    print("\n🔄 Scenario Changes:")
    print(f"  Improved:        {summary['scenarios_improved']} scenarios")
    print(f"  Degraded:        {summary['scenarios_degraded']} scenarios")
    print(f"  Unchanged:       {comparison_results['baseline']['num_scenarios'] - summary['scenarios_improved'] - summary['scenarios_degraded']} scenarios")

    # Show which scenarios improved
    if comparison_results["improved_scenarios"]:
        print(f"\n✅ Scenarios that improved (first 5):")
        for sid in comparison_results["improved_scenarios"][:5]:
            print(f"  - Scenario {sid}")

    if comparison_results["degraded_scenarios"]:
        print(f"\n⚠️ Scenarios that degraded (first 5):")
        for sid in comparison_results["degraded_scenarios"][:5]:
            print(f"  - Scenario {sid}")


# ============================================================
# MAIN EVALUATION PIPELINE
# ============================================================

def run_full_evaluation(final_checkpoint, base_checkpoint):
    """
    Run the complete evaluation pipeline.
    """
    print("\n" + "="*70)
    print("🚀 STARTING FULL EVALUATION PIPELINE")
    print("="*70)

    # Build final trained model
    print("\n📦 Building final model sampler...")
    final_model = build_sampler_from_checkpoint(final_checkpoint)

    # Evaluate on validation set
    print("\n📊 Evaluating trained model on validation set...")
    eval_results = evaluate_agent_on_scenarios(
        sampling_client=final_model,
        tokenizer=tokenizer,
        scenarios=validation_scenarios,
        num_rollouts_per_scenario=3,
        verbose=False,
    )

    # Print report
    print_evaluation_report(eval_results, "Trained Model")

    # Optional: Compare with baseline
    try:
        print("\n📊 Loading baseline for comparison...")
        # Get initial checkpoint (before training)
        # baseline_path = policy_training_client.save_weights_for_sampler(
        #     name=f"email-agent-{RUN_ID}-step-000000"
        # ).result().path

        # baseline_client = service_client.create_sampling_client(
        #     model_path=baseline_path,
        # )


        baseline_client = build_sampler_from_checkpoint(base_checkpoint)

        # Compare on subset for speed
        comparison = compare_models(
            baseline_client=baseline_client,
            trained_client=final_model,
            tokenizer=tokenizer,
            scenarios=validation_scenarios[:20],  # First 20 for speed
            num_rollouts=2,
            verbose=False,
        )

        print_comparison_report(comparison)

    except Exception as e:
        print(f"\n⚠️ Could not compare with baseline: {e}")

    return eval_results


# Usage:


In [38]:
BASE_MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507" #instruct model
final_checkpoint="tinker://f87fddc8-75e4-50cd-a0ad-f603ab570aac:train:0/sampler_weights/email-agent-b0a39866-final"
eval_results_final_vs_base = run_full_evaluation(final_checkpoint=final_checkpoint, base_checkpoint=BASE_MODEL_NAME)


🚀 STARTING FULL EVALUATION PIPELINE

📦 Building final model sampler...

📊 Evaluating trained model on validation set...

EVALUATING ON 40 SCENARIOS


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Creating email DB from Hugging Face dataset...
Loading Enron emails from HF...


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/185M [00:00<?, ?B/s]

...

Creating FTS indexes and triggers...
DB ready with 225965 emails; skipped 57734, duplicates 233702.

📈 Trained Model EVALUATION REPORT

📊 Overall Performance:
  Accuracy:        25.00%
  Accepted:        10/40
  Rejected:        30/40

❌ Failure Breakdown:
  Wrong Answer: 30 (100.0%)

⏱️ Performance:
  Total time:      2477.5s
  Avg per scenario: 61.94s

✅ Sample Successes:

  Q: What are the positions EES is considering regarding PG&E and the retail rate cap...
  A: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"...

  Q: What was the outcome of FERC's order on the AEP/CSW merger?...
  A: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"...

  Q: Has the ISDA been signed for the Xerox deal?...
  A: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"...

❌ Sample Failures:

  Q: What is the name of PentaSafe's new security software and when is it expected to...
  Expected: VigilEnt Security Age

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


📊 Evaluating Trained Model...

EVALUATING ON 20 SCENARIOS


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


🔄 MODEL COMPARISON REPORT

📊 Accuracy Comparison:
  Baseline:        5.00%
  Trained:         25.00%
  Improvement:     +20.0%

📈 Acceptance Comparison:
  Baseline:        1 accepted
  Trained:         5 accepted
  Net change:      +4

🔄 Scenario Changes:
  Improved:        5 scenarios
  Degraded:        1 scenarios
  Unchanged:       14 scenarios

✅ Scenarios that improved (first 5):
  - Scenario 4764
  - Scenario 3752
  - Scenario 845
  - Scenario 5007
  - Scenario 3471

⚠️ Scenarios that degraded (first 5):
  - Scenario 1413


In [39]:
eval_results_final_vs_sft = run_full_evaluation(final_checkpoint='tinker://f87fddc8-75e4-50cd-a0ad-f603ab570aac:train:0/sampler_weights/email-agent-b0a39866-final',
                                  base_checkpoint='tinker://f87fddc8-75e4-50cd-a0ad-f603ab570aac:train:0/sampler_weights/email-agent-sft-warmup-5b2329a2')


🚀 STARTING FULL EVALUATION PIPELINE

📦 Building final model sampler...

📊 Evaluating trained model on validation set...

EVALUATING ON 40 SCENARIOS


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]


📈 Trained Model EVALUATION REPORT

📊 Overall Performance:
  Accuracy:        20.00%
  Accepted:        8/40
  Rejected:        32/40

❌ Failure Breakdown:
  Wrong Answer: 32 (100.0%)

⏱️ Performance:
  Total time:      2366.2s
  Avg per scenario: 59.15s

✅ Sample Successes:

  Q: What are the positions EES is considering regarding PG&E and the retail rate cap...
  A: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"...

  Q: What was the outcome of FERC's order on the AEP/CSW merger?...
  A: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"...

  Q: Has the ISDA been signed for the Xerox deal?...
  A: Do NOT add explanations or commentary.

{"tool": "search_inbox", "arguments": {"...

❌ Sample Failures:

  Q: What is the name of PentaSafe's new security software and when is it expected to...
  Expected: VigilEnt Security Agent, expected to be available to the general market in the f...
  Got: Do NOT add explanations or comm

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


📊 Evaluating Trained Model...

EVALUATING ON 20 SCENARIOS


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


🔄 MODEL COMPARISON REPORT

📊 Accuracy Comparison:
  Baseline:        20.00%
  Trained:         25.00%
  Improvement:     +5.0%

📈 Acceptance Comparison:
  Baseline:        4 accepted
  Trained:         5 accepted
  Net change:      +1

🔄 Scenario Changes:
  Improved:        1 scenarios
  Degraded:        0 scenarios
  Unchanged:       19 scenarios

✅ Scenarios that improved (first 5):
  - Scenario 3471
